# Introduction
In 05_hyperparameter_tuning, I performed 5-fold Cross Validation, and picked the best hyperparameters and the score from the best fold. However it has introduced optimisim as the best fold might be randomly looking best, since there are noises present in the dataset. An alternative way is to use nested CV where the inner loop does the hyperparameter tuning and the outer loop does the performance estimation

# Research Question: 
**Does K-fold cross-validation overestimate model performance compared to nested cross-validation in this tabular regression setting ?**

# Method
1. Define Outer 3 fold cross validation and within each fold of OuterCV, define 3 fold inner cross validation
2. Perform hyperparameter tuning on innerCV and select the best set of hyperparameter
3. With the set of best performing hyperparameters, train on all outer training data, and evaluate on the test set of that outer CV fold 
4. Repeat for all the other folds 

The model used will be the best scoring model LightGBM

# Exprimental Setup
- Outer 3-fold CV
- Inner 3-fold CV 
- Metric: RMSE, RMSE STD (same as during 05_hyperparameter_tuning)
- Same processing pipeline with 05_hyperparameter_tuning as controlled variable for a fair comparson

# Setup

In [2]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

In [8]:
# Same data transformation as before
import numpy as np
import pandas as pd
import joblib

from src.data import load_train_data
from src.features import add_engineered_features

data = load_train_data()
X = add_engineered_features(data)
y = np.log1p(data['SalePrice'])
X = X.drop(columns=['SalePrice'])

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

nominal_cols = ['MSZoning','MSSubClass', 'Street',  'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType','Foundation', 'Heating','CentralAir', 'Electrical','Functional',
        'GarageFinish','PavedDrive',
       'SaleType', 'SaleCondition'
]

ordinal_cols = ['OverallQual','OverallCond','ExterQual', 'ExterCond',
                'HeatingQC','KitchenQual'

]
num_structured_missing = ['GarageYrBlt']
num_unstruc_missing = ['LotFrontage']

rest_num_cols = [
  col for col in X.columns
  if col not in nominal_cols + ordinal_cols + ["SalePrice"] + num_structured_missing + num_unstruc_missing
]

# pipelines

# for numerical attributes with structured missingness, value 0 will be imputed 
num_struc_missing_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='constant',
   fill_value = 0)),
  ('scaler',StandardScaler())
])

# for numerical attributes with true missingness, and the rest of the num cols, the median value will be imputed 
rest_num = Pipeline([
  ('imputer',
   SimpleImputer(strategy='median')),
   ('scaler',StandardScaler())
])


# ordinal pipelines improved
num_ord_cols = ['OverallQual','OverallCond']
cat_ord_cols = ['ExterQual', 'ExterCond','HeatingQC','KitchenQual']
num_ordinal_pipeline = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('scaler',StandardScaler())
])

qual_scale = ['None', 'Po','Fa','TA','Gd','Ex']
cat_ordinal_pipeline = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('encoder', OrdinalEncoder(
    categories=[qual_scale] * len(cat_ord_cols),
    handle_unknown= 'use_encoded_value',
    unknown_value=-1
  ))
])

# preprocessor
preprocessor = ColumnTransformer([
  ('num_struc',num_struc_missing_pipeline,
    num_structured_missing),
  ('rest_num', rest_num, 
   rest_num_cols),
  ('nominal', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols),
  ('num_ordinal', num_ordinal_pipeline, num_ord_cols),
  ('cat_ordinal', cat_ordinal_pipeline, cat_ord_cols),
])

# Nested CV

In [9]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

outer_cv = KFold(n_splits=3, shuffle=True, random_state=42)

In [13]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

In [14]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV

pipeline_lgb = Pipeline([
  ('preprocessor', preprocessor),
  ('model',LGBMRegressor(random_state=42))
])

param_dist = {
  'model__n_estimators': [100, 300, 500],
  'model__learning_rate': [0.01, 0.05, 0.1],
  'model__num_leaves': [31, 63, 127],
  'model__max_depth': [-1,10,20],
  'model__min_child_samples':[5,10,20] 
}

outer_cv_scores = []

for train_index, test_index in outer_cv.split(X):
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

  lgb_search = RandomizedSearchCV(
    estimator = pipeline_lgb,
    param_distributions= param_dist,
    n_iter = 30, 
    scoring = "neg_root_mean_squared_error",
    cv = inner_cv,
    n_jobs = -1,
    random_state=42,
    verbose=0
  )

  lgb_search.fit(X_train, y_train)
  best_model = lgb_search.best_estimator_

  preds = best_model.predict(X_test)
  rmse = np.sqrt(mean_squared_error(y_test, preds))

  outer_cv_scores.append(rmse)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001211 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2657
[LightGBM] [Info] Number of data points in the train set: 973, number of used features: 138
[LightGBM] [Info] Start training from score 12.028454
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [15]:
print("outer cv scores:", outer_cv_scores)

mean_nested = np.mean(outer_cv_scores)
std_nested = np.std(outer_cv_scores)

print("Nested CV RMSE mean:", mean_nested)
print("Nested CV RMSE std:", std_nested)


outer cv scores: [np.float64(0.13719776395674474), np.float64(0.15343926649383147), np.float64(0.1223454536313562)]
Nested CV RMSE mean: 0.1376608280273108
Nested CV RMSE std: 0.01269821826955118


# Results 

|Method|Mean RMSE|RMSE STD|
|------|---------|--------|
|5-fold cross validation|0.130961|0.010407|
|Nested 3*3 cross validation|0.137661|0.012698|


(0.137661 - 0.130961) / 0.130961 = 5.12% 

# Conclusion

The nested cross validation produced a higher mean RMSE (0.137661) compared to standard 5-fold cross-validation (0.13096), error was underestimated by approximately 5.12%. This indicates that the original estimate was optimistically biased because hyperparameters were selected winthin the same validation folds

The increase in RMSE standard deviation also suggests that the variance of model was underestimated under standard cross validation

Thus this experiment demonstrates that nested cross-validation provides a more reliable estimate of generalisation performance in this tabular regression setting